# BYU — Locating Bacterial Flagellar Motors 2025

## Solution Notebook

**Author**: Ahmed Dawod &middot; **Final score**: private **0.6069** / public **0.5974** &middot; **Code**: [github.com/Ahmedn1/byu-flagellar-motors-2025](https://github.com/)

### TL;DR

A 2D U-Net (resnet34 encoder, 7-channel z-context) trained with BCE + `pos_weight=200` on tomograms resampled to 32 Å/voxel. At inference: 3-way test-time augmentation (identity + horizontal flip + vertical flip), heatmaps averaged, 3D Gaussian smooth, global argmax → motor coordinate. Both Kaggle T4 GPUs are used in parallel via a thread queue, which is what makes the TTA budget fit under the 12-hour cap.

### What this notebook does

1. Auto-discovers the test directory and per-tomogram voxel spacings from the competition data.
2. Loads a TorchScript-traced U-Net (`model_a.ts`) onto each available GPU.
3. For each test tomogram, runs three forward passes (identity + h-flip + v-flip), averages the heatmaps, smooths, takes the global 3D argmax.
4. Applies a presence threshold (val-tuned to 0.65). Writes `submission.csv`.

**Setup**: attach a Kaggle Dataset containing `model_a.ts` + `model_a.meta.json` (built locally from the training checkpoint via `submission_assets/build_traced_model.py`). Internet off, GPU = T4 ×2.

## Problem framing

A **tomogram** is a 3D image reconstructed from a tilt-series of 2D projections of bacteria flash-frozen in ice (cryo-electron tomography). Each tomogram is supplied as a directory of JPEG slices. The task is to find the (z, y, x) coordinate of a **flagellar motor** — a ~25 nm molecular machine — if one is present, or abstain (`-1, -1, -1`) if not. About half of the test tomograms contain no motor.

The metric is **F<sub>β=2</sub>** with a **τ = 1000 Å** Euclidean tolerance: a prediction is a true positive if it lands within 1000 Å of the ground-truth motor center (in *physical* distance — voxel-distance is multiplied by the per-tomogram voxel spacing). β=2 means recall is weighted ~4× as heavily as precision: missing a real motor hurts much more than a false alarm on an empty tomogram. The 1000 Å tolerance is quite forgiving — a motor's apparent size at 16 Å/voxel is ~25 voxels, and 1000 Å is ~62 voxels, so we don't need pixel-perfect localization to score TP.

## Solution journey & key decisions

I tried three approaches before settling on this one:

1. **2D U-Net heatmap detector** (this notebook) — best val F (0.889) and best private (0.607).
2. **3D U-Net heatmap regression** (BasicUNet, 96×192×192 crops, sliding-window inference) — val F=0.842, but inference was prohibitively slow on Kaggle T4 (~30+ hours estimated for the full test set) so it never made it to a final submission as a standalone model.
3. **3D classifier + presence head** — collapsed to "always predict positive" due to the per-window class-balance mismatch between training and inference distributions.

Within the 2D approach, the choices that actually moved the needle:

- **BCE with `pos_weight=200`** beat CenterNet focal loss by **+0.77 F** on val. Focal loss is designed for cases with thousands of objects per image; here it had ~1 positive pixel per 150k negatives and the negative-loss term collapsed to ~0 gradient, so the model satisfied itself by predicting zero everywhere. Pos-weighted BCE keeps gradient flowing on the rare positive pixel.
- **7-channel z-context input** (slices `[z-3, z-2, z-1, z, z+1, z+2, z+3]` stacked as channels) over 3 channels gave **+0.018 F** — recovered 4 motors that were too faint in a single slice. The first conv layer mixes the z-neighbors; the rest of the model is still 2D but with richer initial features.
- **Denser positive sampling**: `pos_per_tomo=8` (8 slices near each motor per epoch) over 4 gave **+0.015 F**. Effectively more gradient on the rare positive class per epoch, similar in spirit to but compositional with the `pos_weight` boost.
- **3-way TTA** (identity + h-flip + v-flip), no `both` flip: **+0.011 F** on val over single-pass (0.889 → 0.910). The 4-way version (adding `h+v`) was *worse* on val — the both-flip output was a near-duplicate of identity and dragged the average back toward noise.
- **2-GPU threading**: each Kaggle T4 hosts its own copy of the model; a shared `queue.Queue` of tomo IDs is consumed by one thread per GPU. PyTorch releases the GIL during CUDA ops, so the threads run in true parallel. This halves wall time → 3-way TTA fits under the 12 h Kaggle cap.

### What didn't work (so the next person doesn't redo it)

- **Weight averaging (SWA, EMA, greedy soup)**: all within seed noise of baseline. Diagnosed cause: cosine LR decays to nearly zero by the last few epochs, so late-stage checkpoints sit in near-identical narrow basins — averaging discrete near-identical points doesn't move toward a wider minimum. SWA with the proper SWALR scheduler (constant high LR over a snapshot phase) might still work but wasn't worth another retrain after the failures.
- **Heavy augmentation** (gamma + intensity + noise + zoom): hurt val by 0.044. Confirmed to also hurt private (-0.116) when submitted.
- **MixUp** (α=0.2): blurred decision boundaries, FPs nearly doubled.
- **Larger backbone (resnet50)**: better recall, much worse precision. Net F drop of 0.026.
- **Multi-task aux head** (presence classifier on bottleneck): joint loss was hard to balance; even at the right λ the model landed within seed noise of baseline.
- **Multi-scale inference** (28 Å + 32 Å): val F same, FPs cut in half — but the smaller-spacing scale increased per-tomo compute by 1.49× and the submission timed out at 12 h.
- **3D cascade** (route uncertain 2D tomos to a 3D model for second opinion): scored 0.385 on private. The 3D model's val-to-test generalization gap was much wider than the 2D model's, so trusting it on the hardest cases backfired.

Full experiment log (40+ experiments): see the `EXPERIMENTS.md` and `.lab/` directory in the [GitHub repo](https://github.com/).

In [ ]:
import json, sys, time, gc, traceback, threading, queue
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.ndimage import zoom, gaussian_filter

print('torch', torch.__version__)
n_gpus = torch.cuda.device_count()
print(f'GPU count: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
INPUT_ROOT = Path('/kaggle/input')

# --- Locate model files (uploaded as a Kaggle Dataset).
model_files = list(INPUT_ROOT.rglob('model_a.ts'))
assert model_files, 'model_a.ts not found in /kaggle/input — attach the model dataset.'
MODEL_PATH = model_files[0]
META = json.loads((MODEL_PATH.parent / 'model_a.meta.json').read_text())
print('model:', MODEL_PATH)
print('meta :', META)

# --- Locate the competition test directory (the deepest /kaggle/input/.../test/ with subdirs).
candidate_test_dirs = []
for p in INPUT_ROOT.rglob('test'):
    if p.is_dir() and any(c.is_dir() for c in p.iterdir()):
        candidate_test_dirs.append(p)
assert candidate_test_dirs, 'no test/ directory found under /kaggle/input'
TEST_DIR = max(candidate_test_dirs, key=lambda p: sum(1 for _ in p.iterdir()))
tomo_ids = sorted([p.name for p in TEST_DIR.iterdir() if p.is_dir()])
print(f'test dir: {TEST_DIR}  ({len(tomo_ids)} tomos)')

# --- Per-tomogram voxel spacing (Å/voxel) — needed to map predictions back to the
#     original voxel frame. We scan every CSV under /kaggle/input for one with a
#     'Voxel spacing' column; the competition data ships it inline with train_labels.csv.
spacing_map = {}
for csv_path in INPUT_ROOT.rglob('*.csv'):
    try:
        df = pd.read_csv(csv_path, nrows=1)
    except Exception:
        continue
    cols_lc = [c.lower() for c in df.columns]
    if any('voxel spacing' in c for c in cols_lc) and 'tomo_id' in cols_lc:
        full = pd.read_csv(csv_path)
        sp_col = [c for c in full.columns if 'voxel spacing' in c.lower()][0]
        tid_col = [c for c in full.columns if c.lower() == 'tomo_id'][0]
        for _, r in full.iterrows():
            spacing_map[str(r[tid_col])] = float(r[sp_col])
        print(f'pulled spacings from: {csv_path}')
for t in tomo_ids:
    spacing_map.setdefault(t, 16.0)

# --- Hyperparameters from the meta file (and one hardcoded fallback).
IN_CH = int(META['in_channels'])
TARGET_SPACING = float(META.get('target_spacing') or 32.0)
SMOOTH_SIGMA = float(META.get('peak_smooth_sigma', 1.0))
THRESHOLD = float(META.get('presence_threshold', 0.65))
TTA_AUGS = META.get('tta_augs', ['identity', 'hflip', 'vflip'])
print(f'IN_CH={IN_CH}  TARGET_SP={TARGET_SPACING}  THR={THRESHOLD}  SMOOTH={SMOOTH_SIGMA}')
print(f'TTA={TTA_AUGS}  ({len(TTA_AUGS)} passes/tomo)')
print(f'will distribute {len(tomo_ids)} tomos across {n_gpus} GPU(s)')

In [ ]:
# --- Inference helpers.

def load_tomo(tomo_dir: Path) -> np.ndarray:
    """Stack a directory of slice_*.jpg files into a uint8 (Z, H, W) volume."""
    slices = sorted(tomo_dir.glob('slice_*.jpg')) or sorted(tomo_dir.glob('*.jpg'))
    return np.stack([np.array(Image.open(p).convert('L'), dtype=np.uint8) for p in slices], axis=0)


def resample_vol(vol: np.ndarray, orig_sp: float, target_sp: float):
    """Isotropically resample by factor orig_sp/target_sp on all 3 axes.
    Brings every tomogram to the same physical voxel size (32 Å), so a motor's
    apparent voxel-size is constant across tomos."""
    if abs(orig_sp - target_sp) < 1e-6:
        return vol, 1.0
    f = orig_sp / target_sp
    out = zoom(vol, (f, f, f), order=1)
    if out.dtype != vol.dtype:
        out = np.clip(out, 0, 255).astype(np.uint8)
    return out, f


def apply_aug(vol: np.ndarray, aug: str) -> np.ndarray:
    """Spatial TTA — applied to the input volume *and* to the output heatmap as the inverse.
    All three augs are self-inverse: a second application undoes the first."""
    if aug == 'identity': return vol
    if aug == 'hflip':    return vol[:, :, ::-1].copy()
    if aug == 'vflip':    return vol[:, ::-1, :].copy()
    raise ValueError(aug)

invert_aug = apply_aug


@torch.no_grad()
def predict_volume_once(model, vol: np.ndarray, device: str,
                         in_channels: int = 7, batch: int = 8) -> np.ndarray:
    """Build a 7-channel z-stack per slice z = [z-3..z+3] (edge-clipped) and run
    the model on every slice. Returns (Z, H, W) sigmoid heatmap stack."""
    Z, H, W = vol.shape
    out = np.zeros((Z, H, W), dtype=np.float32)
    half = in_channels // 2
    for z0 in range(0, Z, batch):
        z1 = min(Z, z0 + batch)
        batch_imgs = []
        for z in range(z0, z1):
            zs = [min(max(0, z + d), Z - 1) for d in range(-half, -half + in_channels)]
            batch_imgs.append(np.stack([vol[zi] for zi in zs]).astype(np.float32) / 255.0)
        x = torch.from_numpy(np.stack(batch_imgs)).to(device)
        out[z0:z1] = torch.sigmoid(model(x)).squeeze(1).cpu().numpy()
    return out


def extract_peak_3d(heat: np.ndarray):
    """Global 3D argmax + its value."""
    flat = int(np.argmax(heat))
    z, y, x = np.unravel_index(flat, heat.shape)
    return (int(z), int(y), int(x)), float(heat[z, y, x])

In [ ]:
# --- Load one model instance per GPU. Threads share the dict.
MODELS = {}
for gid in range(max(1, n_gpus)):
    dev = f'cuda:{gid}' if n_gpus > 0 else 'cpu'
    m = torch.jit.load(str(MODEL_PATH), map_location=dev)
    m.eval()
    MODELS[gid] = (m, dev)
print(f'loaded model on {len(MODELS)} GPU(s)')


def process_tomo_on(gpu_id: int, tid: str):
    """Full inference for one tomogram on the model assigned to gpu_id.
    Thread-safe because each GPU has its own model instance and CUDA stream."""
    model, device = MODELS[gpu_id]
    vol = load_tomo(TEST_DIR / tid)
    sp = spacing_map[tid]
    vol_rs, scale = resample_vol(vol, sp, TARGET_SPACING)
    # 3-way TTA: identity + hflip + vflip, average the heatmaps in the resampled frame.
    accum = None
    for aug in TTA_AUGS:
        vol_a = apply_aug(vol_rs, aug)
        heat_a = predict_volume_once(model, vol_a, device, IN_CH, 8)
        heat = invert_aug(heat_a, aug)
        accum = heat if accum is None else accum + heat
    heat = accum / len(TTA_AUGS)
    if SMOOTH_SIGMA > 0:
        heat = gaussian_filter(heat, (SMOOTH_SIGMA, SMOOTH_SIGMA, SMOOTH_SIGMA))
    (z, y, x), peak = extract_peak_3d(heat)
    # back-transform from resampled frame to original voxel frame for the submission.
    return tid, float(peak), z / scale, y / scale, x / scale

In [ ]:
# --- Sanity check on one tomogram before launching the full run.
if len(tomo_ids) > 0:
    tid = tomo_ids[0]
    t0 = time.time()
    _, peak, zo, yo, xo = process_tomo_on(0, tid)
    print(f'sanity ({tid}): peak={peak:.3f} at ({zo:.0f}, {yo:.0f}, {xo:.0f}) — {time.time()-t0:.1f}s')

In [ ]:
# --- Threaded inference: one worker thread per GPU pulls from a shared queue.
work_q = queue.Queue()
for tid in tomo_ids:
    work_q.put(tid)

results_lock = threading.Lock()
all_results = []
failures = []
times = []
done_count = [0]


def gpu_worker(gpu_id):
    while True:
        try:
            tid = work_q.get(timeout=1)
        except queue.Empty:
            return
        t_start = time.time()
        try:
            tid_, peak, zo, yo, xo = process_tomo_on(gpu_id, tid)
            with results_lock:
                all_results.append((tid_, peak, zo, yo, xo))
        except Exception as e:
            with results_lock:
                failures.append((tid, str(e)))
                all_results.append((tid, -1.0, -1.0, -1.0, -1.0))
            print(f'[gpu{gpu_id}] {tid} FAIL: {e}', flush=True)
            traceback.print_exc()
        gc.collect()
        if torch.cuda.is_available():
            with torch.cuda.device(gpu_id):
                torch.cuda.empty_cache()
        dt = time.time() - t_start
        with results_lock:
            done_count[0] += 1
            times.append(dt)
            if done_count[0] % 25 == 0 or done_count[0] == len(tomo_ids):
                slowest = sorted(times, reverse=True)[:3]
                print(f'  {done_count[0]}/{len(tomo_ids)}  '
                      f'(gpu{gpu_id} just did {tid}: {dt:.1f}s)  '
                      f'slowest3={[f"{s:.1f}" for s in slowest]}  '
                      f'failures={len(failures)}', flush=True)
        work_q.task_done()


t0 = time.time()
threads = []
n_workers = max(1, n_gpus)
print(f'spawning {n_workers} worker thread(s)')
for gid in range(n_workers):
    t = threading.Thread(target=gpu_worker, args=(gid,), daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

wall = time.time() - t0
print(f'\nall workers done in {wall/60:.1f} min wall clock')
print(f'total compute time = {sum(times)/60:.1f} min (speedup = {sum(times)/wall:.2f}x)')
print(f'results: {len(all_results)} of {len(tomo_ids)} expected')

# --- Apply threshold and emit submission.csv.
by_id = {r[0]: r for r in all_results}
rows = []
abstain_count = 0
for tid in tomo_ids:  # iterate in canonical order
    if tid in by_id:
        _, peak, zo, yo, xo = by_id[tid]
        if peak < THRESHOLD:
            rows.append((tid, -1.0, -1.0, -1.0))
            abstain_count += 1
        else:
            rows.append((tid, zo, yo, xo))
    else:
        rows.append((tid, -1.0, -1.0, -1.0))
        abstain_count += 1
sub = pd.DataFrame(rows, columns=['tomo_id', 'Motor axis 0', 'Motor axis 1', 'Motor axis 2'])
sub.to_csv('submission.csv', index=False)
print(f'\nsubmission.csv: {len(sub)} rows, {abstain_count} abstain, {len(sub)-abstain_count} predicted')
print(f'failures: {len(failures)}')
sub.head()

## Reproducing this notebook

Everything needed to train the model and rebuild `model_a.ts` is in the [companion repo](https://github.com/) under the same structure:

1. **Preprocess** the tomograms to 32 Å/voxel uint8 caches:
   ```bash
   PYTHONPATH=. python scripts/preprocess_all.py
   ```
2. **Build the train/val split** (stratified 80/20 by motor-count × spacing bucket):
   ```bash
   PYTHONPATH=. python scripts/build_split.py
   ```
3. **Train** (single A100, ~70 minutes for 10 epochs):
   ```bash
   PYTHONPATH=. python scripts/run_2d.py --cfg configs/recipe.py --exp_dir runs/run0
   ```
   Produces `runs/run0/ckpt_best.pt` and prints a final `F_BETA=<val>` line.
4. **Trace to TorchScript** so this notebook needs no `segmentation_models_pytorch` at submit time:
   ```bash
   python submission_assets/build_traced_model.py --ckpt runs/run0/ckpt_best.pt
   ```
   Produces `submission_assets/model_a.ts` + `submission_assets/model_a.meta.json`.
5. **Submit**: create a Kaggle Dataset containing those two files, attach it to a fresh Kaggle notebook from the competition page, paste in this notebook's content, Save & Run All, then Submit to Competition.

## Acknowledgements

Thanks to BYU and Kaggle for hosting the competition, to [Bartley](https://www.kaggle.com/brendanartley) for releasing a public external dataset that everyone in the top tier used (which I didn't but should have), and to the MIC-DKFZ team for a writeup that taught me more about 3D segmentation than the whole competition runtime did.